## Gene panel statistics 

In [42]:
import pandas as pd
import numpy as np
import os
import anndata as ad
import squidpy as sq
import scanpy as sc
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

In [9]:
print(os.getcwd())
this_path = os.getcwd()

C:\Users\Chenxi Zhou\Desktop\Neural Crest\SingleCrest\merscope\result\analysis\2025MAY\Tutorial_Github\tutorial


In [43]:
adata = ad.read_h5ad("MER02_R1_filtered_stage22_subset.h5ad")
adata.X = adata.layers["raw_counts"]

path_save = str(os.getcwd()) + "//gene_panel_QC//"
Path.mkdir(Path(path_save), parents=True, exist_ok = True)


## statistics of gene expression

In [44]:

## statistics of gene expression matrix
adata.X = adata.layers["raw_counts"]
stat = sc.pp.calculate_qc_metrics(adata,percent_top=(1,5,10,15))
stat[0].to_csv(path_save + "cell_stat.csv")  
stat[1].to_csv(path_save + "gene_stat.csv")



In [45]:
## frequency of single transcript event (where only one transcript of a gene is detected in a cell)

from scipy.sparse import issparse

expr_matrix = adata[:, adata.var_names].X

if issparse(expr_matrix):
    expr_matrix = expr_matrix.toarray()

# create gene_by_cell expression dataFrame
gene_df = pd.DataFrame(
    expr_matrix,
    columns=adata.var_names,
    index=adata.obs_names
)

# count the frequency of single transcript event
gene_l = np.where(gene_df.to_numpy() == 1, 1,0).sum(axis=0) 

count_1 = pd.DataFrame(gene_l, index = adata.var_names)
count_1.to_csv(path_save + "single_transcript_event.csv")

## Evaluate the spatial specificity

In [46]:
## Evaluate the spatial specificity  by Morans'I

# normalization and log1 transform of the data
# normalization by volume
adata.obs["volume"]  
adata.obs["volume_scaled"]=adata.obs["volume"]/1000   # scaled the volume by 1000 so the normalized count will above 1
adata.obs["volume_scaled"]

if (adata.obs["volume"] <= 0).any():
    raise ValueError("Volume values must be positive and non-zero!")  # Check if volume values are valid (non-zero to avoid division by zero)
    
adata.layers["normal_volume"] = adata.layers["raw_counts_bg_removeby1"] / adata.obs["volume_scaled"].values[:, None]

## log transformation

adata.layers["log1p_normal_volume"]=adata.layers["normal_volume"].copy()
sc.pp.log1p(adata, base=2, copy=False, chunked=None, chunk_size=None, layer="log1p_normal_volume", obsm=None)

# calculate the Moran's I 

adata.X = adata.layers["log1p_normal_volume"]

sq.gr.spatial_neighbors(adata, coord_type="generic")
sq.gr.spatial_autocorr(adata, mode="moran")
adata.uns["moranI"].to_csv(path_save + "moranI.csv")

adata.uns["moranI"]


,I,pval_norm,var_norm,pval_norm_fdr_bh
MYL1.L,0.931775,0.000000e+00,0.000047,0.000000e+00
ACTC1.L,0.924622,0.000000e+00,0.000047,0.000000e+00
MYOD1.S,0.895542,0.000000e+00,0.000047,0.000000e+00
PITX1.L,0.889972,0.000000e+00,0.000047,0.000000e+00
RAX.L,0.850988,0.000000e+00,0.000047,0.000000e+00
SOX3.S,0.848063,0.000000e+00,0.000047,0.000000e+00
SOX2.L,0.843485,0.000000e+00,0.000047,0.000000e+00
SOX10.L,0.818424,0.000000e+00,0.000047,0.000000e+00
HOXB9.S,0.785770,0.000000e+00,0.000047,0.000000e+00
SOX8.L,0.783968,0.000000e+00,0.000047,0.000000e+00


In [47]:
## draw normalized gene expression on sections

fig_save = Path(path_save + "\\spatial_expression\\")
Path(fig_save).mkdir(parents=True,exist_ok=True)
for gene in adata.var_names:
    fig, ax = plt.subplots( figsize=(12*1, 6*1) )
    sq.pl.spatial_scatter(
        adata,
        color=gene,
        cmap="gray_r",       # optional: change to 'viridis', 'plasma', etc.
        size=10,
        #return_fig=True,
        ax = ax,
        #show=False,
        shape = None,          # prevent pop-up
    )
    
    # Save to file
    plt.savefig(str(fig_save) + "\\" +  f"marker_{gene}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)  # Close to save memory